# C2 real run on Colab (tasks C2-T6 and C2-T7, addendum A2)

This notebook makes the pre-registered C2 run on a **Colab CPU runtime** (Linux), as addendum A2 fixes:
- Python 3.13.0 and the 13 exact package versions of `c2_requirements.txt`;
- XGBoost with 2 threads per fit;
- the frozen code only.

**Before the 2-3 hour run, a pre-flight check of about 8 minutes:**
1. the bundle is undamaged and holds the frozen code;
2. the environment equals the pinned one;
3. the C2 tests pass;
4. on the made-up noise target, this machine reproduces the Linux reference of the platform test byte for byte:
   - the dry run;
   - the full-size model fingerprints.

If any check fails, the notebook stops **before** the real run and says why. Nothing real has been computed at that
point.

**Steps**
1. Upload the new `c2_colab_bundle.zip` to the top level of *My Drive*. Delete the old one first, so Drive does not
   rename the new one.
2. In the first code cell, `RUN = 'T6'` for the real run. Later, `RUN = 'T7'` for the rerun, in a new session.
3. *Runtime → Change runtime type → CPU* (High-RAM on, if offered), then *Runtime → Run all*. Allow Drive access.
4. With Colab Pro+ background execution you can close the browser tab.
   - If Colab disconnects, open the notebook and *Run all* again. Finished fits are kept on Drive
     (`My Drive/c2_runs/<RUN>/c2x_parts`), so the run resumes where it stopped.
   - Each session repeats the pre-flight check, so a new machine is checked too.
5. At the end the notebook prints the verdict and saves `My Drive/c2_runs/<RUN>_results.zip`. Send that file.

Never reuse a run folder: a finished run is never run again (pre-registration section 13).

In [1]:
# 1. Settings, the bundle and the checks of its files
RUN = 'T6'          # 'T6' = the real run; 'T7' = its rerun, in a new session (addendum A2.2)
import hashlib, json, os, re, shutil, subprocess, sys, time, zipfile
from google.colab import drive
assert RUN in ('T6', 'T7'), "RUN must be 'T6' or 'T7'"
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
BUNDLE = f'{DRIVE}/c2_colab_bundle.zip'
ROOT = '/content/c2'                         # the unpacked bundle (fresh every session)
PRE = '/content/c2_preflight'                # this session's pre-flight runs (fresh every session)
OUT = f'{DRIVE}/c2_runs/{RUN}'               # results and resume parts, kept on Drive across sessions
for d in (ROOT, PRE):
    shutil.rmtree(d, ignore_errors=True)
with zipfile.ZipFile(BUNDLE) as zf:
    zf.extractall(ROOT)
with open(f'{ROOT}/bundle_manifest.json') as f:
    man = json.load(f)


def sha(path):
    with open(path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()


def rule(path):
    """Pre-registration hash rule: text files after CRLF -> LF, binary files as they are."""
    with open(path, 'rb') as f:
        b = f.read()
    if path.endswith(('.py', '.md', '.csv', '.txt', '.json')):
        b = b.replace(b'\r\n', b'\n')
    return hashlib.sha256(b).hexdigest()


bad = [f for f, h in man['files'].items() if sha(f'{ROOT}/{f}') != h]
assert not bad, f'the bundle was damaged in transfer: upload it again ({bad[:3]})'
FROZEN, IN_BUNDLE = {}, {}
if man.get('frozen'):
    with open(f"{ROOT}/{man['freeze_file']}") as f:
        FROZEN = {p: h for h, p in (l.split() for l in f if l.strip() and not l.startswith('#'))}
    IN_BUNDLE = {p: h for p, h in FROZEN.items() if os.path.isfile(f'{ROOT}/{p}')}   # not this notebook itself
    changed = [p for p, h in IN_BUNDLE.items() if rule(f'{ROOT}/{p}') != h]
    assert not changed, f'frozen files differ from the freeze record: {changed}'
GPU = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
assert not GPU, 'addendum A2: the real run is on a CPU runtime. Runtime -> Change runtime type -> CPU, then Run all'
E1V = f'{ROOT}/Implementation/revision_1/e1_verification'
os.makedirs(OUT, exist_ok=True)
SESSION = 1 + len([d for d in os.listdir(OUT) if d.startswith('session_')])
LOGS = f'{OUT}/session_{SESSION:02d}'        # this session's logs and checks, on Drive
os.makedirs(LOGS)
print(f"{len(man['files'])} files unpacked and undamaged; frozen code: "
      + (f'all {len(IN_BUNDLE)} frozen files in the bundle match the freeze record' if FROZEN
         else 'NO (a real run will be refused)'))
print(f'run {RUN}: results in {OUT}; session {SESSION}')


def run(*args, log):
    """Run a command in e1_verification, show its output live, keep it in LOGS/log, and stop if it fails."""
    args = [str(a) for a in args]
    print('$', ' '.join(args), flush=True)
    p = subprocess.Popen(args, cwd=E1V, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    with open(f'{LOGS}/{log}', 'a') as f:
        f.write('$ ' + ' '.join(args) + '\n')
        for line in p.stdout:
            print(line, end='', flush=True)
            f.write(line)
            lines.append(line)
    if p.wait() != 0:
        raise RuntimeError(f'step failed (exit {p.returncode}); see the output above')
    return ''.join(lines)

Mounted at /content/drive
62 files unpacked and undamaged; frozen code: all 10 frozen files in the bundle match the freeze record
run T6: results in /content/drive/MyDrive/c2_runs/T6; session 1


In [2]:
# 2. A clean Python 3.13.0 environment with the exact package versions (Colab's own Python is not used)
VENV, VPY = '/content/c2venv', '/content/c2venv/bin/python'
shutil.rmtree(VENV, ignore_errors=True)
os.environ.pop('UV_SYSTEM_PYTHON', None)     # Colab sets it; it does not apply to a separate environment
REQ = f"{ROOT}/{man['requirements']}"
run(sys.executable, '-m', 'pip', 'install', '-q', 'uv', log='environment.log')
run(sys.executable, '-m', 'uv', 'venv', '--python', man['python'], VENV, log='environment.log')
run(sys.executable, '-m', 'uv', 'pip', 'install', '--python', VPY, '-r', REQ, log='environment.log')
with open(REQ) as f:
    want = dict(l.strip().split('==') for l in f if l.strip() and not l.startswith('#'))
frozen_env = run(sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', VPY, log='environment.log')
got = dict(l.strip().split('==') for l in frozen_env.splitlines() if '==' in l)
py = run(VPY, '-c', 'import platform; print(platform.python_version())', log='environment.log').strip()
differ = {n: (got.get(n), v) for n, v in want.items() if got.get(n) != v}
print('python', py, '| packages that differ from the pinned versions:', differ or 'none')
assert py == man['python'] and not differ, 'the environment does not match the pinned one'

$ /usr/bin/python3 -m pip install -q uv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 53.4 MB/s eta 0:00:00
$ /usr/bin/python3 -m uv venv --python 3.13.0 /content/c2venv
Using CPython 3.13.0
Creating virtual environment at: /content/c2venv
Activate with: source /content/c2venv/bin/activate
$ /usr/bin/python3 -m uv pip install --python /content/c2venv/bin/python -r /content/c2/Implementation/revision_1/e1_verification/c2_requirements.txt
Using Python 3.13.0 environment at: /content/c2venv
Resolved 14 packages in 646ms
Prepared 14 packages in 12.75s
Installed 14 packages in 35ms
 + charset-normalizer==3.4.4
 + joblib==1.5.3
 + numpy==2.4.2
 + nvidia-nccl-cu12==2.32.3
 + pandas==3.0.1
 + psutil==7.2.2
 + python-dateutil==2.9.0.post0
 + pytz==2026.3.post1
 + scikit-learn==1.8.0
 + scipy==1.17.1
 + six==1.17.0
 + threadpoolctl==3.6.0
 + tzdata==2025.3
 + xgboost==3.2.0
$ /usr/bin/python3 -m uv pip freeze --python /content/c2venv/bin/python
Using Python 3.13.0 environment at: /con

In [3]:
# 3. The machine: processors, the threads XGBoost honours, and the layout (2 threads per fit, addendum A2.1)
REMOVED = {v: os.environ.pop(v, None) for v in ('OMP_NUM_THREADS', 'OMP_THREAD_LIMIT')}   # unset for every run
probe = ('import json, c2_xgb_confirm as rn; '
         'print(json.dumps({"machine": rn.machine(), "cpu": rn.cpu_model(), "logical processors": rn.processors(), '
         '"cgroup CPU quota": rn.cgroup_cpus(), "most threads XGBoost honours": rn.xgb_threads(rn.processors())}))')
hw = json.loads(run(VPY, '-c', probe, log='environment.log').strip().splitlines()[-1])
run(VPY, '-c', 'import numpy; numpy.show_runtime()', log='environment.log')
MAXT = hw['most threads XGBoost honours']
assert MAXT >= 2, f'XGBoost can use only {MAXT} thread here; addendum A2 needs 2. Choose a larger CPU runtime'
PROCS = MAXT // 2                            # processes x 2 threads = the threads XGBoost honours
hw.update({'run': RUN, 'session': SESSION, 'processes': PROCS, 'xgboost n_jobs': 2,
           'OpenMP variables removed': REMOVED, 'bundle sha256': sha(BUNDLE), 'frozen': bool(FROZEN)})
T6_VERDICT = f'{DRIVE}/c2_runs/T6/c2x_verdict.json'
if RUN == 'T7' and os.path.isfile(T6_VERDICT):
    with open(T6_VERDICT) as f:
        t6_machines = json.load(f)['provenance']['fits made on (addendum A2.3)']
    hw['T6 made on'] = t6_machines
    same_cpu = all(m.split(' | ')[0] == hw['cpu'] for m in t6_machines)
    print('T7 processor type ' + ('equals' if same_cpu else 'differs from') + ' T6\'s'
          + (' (addendum A2.2 accepts this when Colab assigns no other type)' if same_cpu else ' (as addendum A2.2 asks)'))
with open(f'{LOGS}/environment.json', 'w') as f:
    json.dump(hw, f, indent=1)
print(json.dumps(hw, indent=1))

$ /content/c2venv/bin/python -c import json, c2_xgb_confirm as rn; print(json.dumps({"machine": rn.machine(), "cpu": rn.cpu_model(), "logical processors": rn.processors(), "cgroup CPU quota": rn.cgroup_cpus(), "most threads XGBoost honours": rn.xgb_threads(rn.processors())}))
{"machine": "AMD EPYC 7B12 | Linux x86_64 | 8 processors", "cpu": "AMD EPYC 7B12", "logical processors": 8, "cgroup CPU quota": -1, "most threads XGBoost honours": 8}
$ /content/c2venv/bin/python -c import numpy; numpy.show_runtime()
[{'numpy_version': '2.4.2',
  'python': '3.13.0 (main, Oct 16 2024, 03:23:02) [Clang 18.1.8 ]',
  'uname': uname_result(system='Linux', node='013811d661ab', release='6.6.122+', version='#1 SMP Thu Apr 30 18:17:14 UTC 2026', machine='x86_64')},
 {'simd_extensions': {'baseline': ['X86_V2'],
                      'found': ['X86_V3'],
                      'not_found': ['X86_V4', 'AVX512_ICL', 'AVX512_SPR']}},
 {'ignore_floating_point_errors_in_matmul': False},
 {'architecture': 'Haswell'

In [4]:
# 4. The C2 test suites (leakage, rules, runner); no real-target test-year prediction is made
for t in ('test_c2x.py', 'test_c2x_verdict.py', 'test_c2_runner.py'):
    run(VPY, f'../tests/{t}', log='tests.log')

$ /content/c2venv/bin/python ../tests/test_c2x.py
    splits by target date as pre-registered; periods {'pre': 4502, 'lock': 1669}
PASS test_windows_splits_periods (6 s)
    32 arms; widths {137: 1, 161: 30, 149: 1}; splits disjoint and complete for every arm
PASS test_arms_and_design (0 s)
    CRLF and LF text hash the same; binary hashed as bytes; all pinned inputs verified
PASS test_hash_rule_and_inputs (0 s)
    5 worked cases; each set maps the 6 own years to 6 pool years; each own year meets all 10
PASS test_pool_rule_worked_cases (0 s)
    15862 MET_PP rows and 15880 MET rows unchanged; every later row changed
PASS test_lookahead (4 s)
    18 windows with target 2019-12-15: MET_PP changed, MET unchanged
PASS test_day_t_plus_1_reaches_only_met_pp (4 s)
    all 31 weather blocks identical; only the pollutant columns of BASE changed
PASS test_no_aqi_or_pollutant_in_weather (9 s)
    20 pool blocks depend only on the pool file; 11 other blocks only on the real file
PASS test_files_k

In [5]:
# 5. Pre-flight check 1 (made-up target): the dry run must equal the Linux reference of the platform test byte for byte
REPORT = []
run(VPY, 'c2x_features.py', log='preflight.log')                  # the feature digests of this machine


def digests(path):
    with open(path) as f:
        return dict(re.findall(r'^(BASE|meta digest|  \S+)\s.*?([0-9a-f]{16})$', f.read(), re.M))


mine = digests(f'{E1V}/outputs/c2_t2_features_summary.txt')
shutil.copy(f'{E1V}/outputs/c2_t2_features_summary.txt', f'{LOGS}/c2_t2_features_summary.txt')
like = [k for k, p in man['linux_features_summaries'].items() if digests(f'{ROOT}/{p}') == mine]
REPORT.append('inputs: BASE and meta digests equal the reference: '
              + str(all(mine[k] == digests(f"{ROOT}/{next(iter(man['linux_features_summaries'].values()))}")[k]
                        for k in ('BASE', 'meta digest')))
              + '; weather digests equal those of: ' + (', '.join(like) or 'neither reference machine (last-bit '
                'logarithm differences; information only, the model check below decides)'))
REF = f"{ROOT}/{man['linux_reference']['dry']}"
run(VPY, 'c2_xgb_confirm.py', '--dry-run', '--seeds', 0, 1, '--processes', PROCS, '--n-jobs', 2,
    '--out', f'{PRE}/dry', log='preflight.log')


def content(path):
    """The verdict without what is expected to differ: the provenance block (machine, code hashes) and the A2 hash,
    which the reference (made before addendum A2) does not carry."""
    with open(path) as f:
        v = json.load(f)
    v.pop('provenance')
    v.pop('addendum_A2_sha256', None)
    return v


same = {f: rule(f'{PRE}/dry/{f}') == rule(f'{REF}/{f}') for f in ('dry_pooled.csv', 'dry_city.csv')}
same['dry_predictions.npz'] = sha(f'{PRE}/dry/dry_predictions.npz') == sha(f'{REF}/dry_predictions.npz')
same['verdict apart from provenance'] = content(f'{PRE}/dry/dry_verdict.json') == content(f'{REF}/dry_verdict.json')
with open(f'{PRE}/dry/dry_verdict.json') as f:
    fits_on = json.load(f)['provenance']['fits made on (addendum A2.3)']
same['all 64 fits recorded on this machine'] = fits_on == {hw['machine']: 64}
REPORT.append(f'pre-flight dry run ({PROCS} processes x 2 threads) against the Linux reference: {same}')
print('\n'.join(REPORT))
with open(f'{LOGS}/preflight.txt', 'w') as f:
    f.write('\n'.join(REPORT) + '\n')
assert all(same.values()), ('PRE-FLIGHT FAILED: this machine does not reproduce the Linux reference. The real run is '
                            f'not started. Send the folder {LOGS}')

$ /content/c2venv/bin/python c2x_features.py
C2-T2 feature build summary (no model fitted; y not summarised)
inputs: pre-registration and addenda A1 and A2 hashes are the approved ones; 30 pinned files and 2 raw ERA5 hashes match Appendix A
windows: 18 cities; train 11569, val 1510, test 6171 (as pre-registered); target = origin + 1 day for every window
finite: BASE and all 31 weather blocks
lookups: real weather on its own date; pool nulls in 2005-2014 on the same calendar day, one pool year per own year in every set; E5 nulls never the own year

test periods: pre-lockdown 4502, lockdown/Unlock 1669
BASE (19250, 137) digest 2b9598ea703bf8fc
meta digest (city, split, origin, target, period) 36f3533b9bf7c48e

weather blocks (shape, digest):
  MET            (19250, 24)  24ce88cd39835912
  MET_PP         (19250, 24)  dc6b5b9b4d7f578b
  POOL_0         (19250, 24)  9ce20e5ce50e9a89
  POOL_1         (19250, 24)  41a49438d8752e37
  POOL_2         (19250, 24)  2a749968fdea3ea1
  POOL_3       

In [6]:
# 6. Pre-flight check 2 (made-up target): full-size models (400 trees) must equal the Linux reference fingerprints
run(VPY, 'c2_xgb_confirm.py', '--timing', '--processes', PROCS, '--n-jobs', 2, '--out', f'{PRE}/timing',
    log='preflight.log')
with open(f'{PRE}/timing/fingerprints_cpu_p{PROCS}_t2.json') as f:
    mine_fp = json.load(f)['arms']
with open(f"{ROOT}/{man['linux_reference']['fingerprints']}") as f:
    ref_fp = json.load(f)['arms']
n_same = sum(mine_fp.get(a) == ref_fp[a] for a in ref_fp)
with open(f'{PRE}/timing/run_log.log') as f:
    timing = [l.split(' ', 2)[2].strip() for l in f if 'TIMING' in l][-1]
REPORT += [f'pre-flight full-size models: identical to the Linux reference in {n_same} of {len(ref_fp)} arms', timing]
print('\n'.join(REPORT[-2:]))
with open(f'{LOGS}/preflight.txt', 'w') as f:
    f.write('\n'.join(REPORT) + '\n')
assert n_same == len(ref_fp), ('PRE-FLIGHT FAILED: full-size models differ from the Linux reference. The real run is '
                               f'not started. Send the folder {LOGS}')
print('PRE-FLIGHT PASSED')

$ /content/c2venv/bin/python c2_xgb_confirm.py --timing --processes 4 --n-jobs 2 --out /content/c2_preflight/timing
TIMING cpu processes 4 x n_jobs 2: 8 fits of 400 trees + SHAP on 300: wall 52.5 s (one fit takes 25.9 s); estimated 1,280 fits: 2.33 h
pre-flight full-size models: identical to the Linux reference in 8 of 8 arms
TIMING cpu processes 4 x n_jobs 2: 8 fits of 400 trees + SHAP on 300: wall 52.5 s (one fit takes 25.9 s); estimated 1,280 fits: 2.33 h
PRE-FLIGHT PASSED


In [7]:
# 7. The real run: 32 arms x 40 seeds = 1,280 fits, resumable from the parts on Drive
if os.path.isfile(f'{OUT}/c2x_verdict.json'):
    print(f'{RUN} finished in an earlier session; it is not run again (pre-registration section 13)')
else:
    assert FROZEN, 'this bundle is not from the frozen code: no real run (tracker ground rule 3)'
    t0 = time.time()
    run(VPY, 'c2_xgb_confirm.py', '--processes', PROCS, '--n-jobs', 2, '--out', OUT, log='run.log')
    print(f'this session: {(time.time() - t0) / 3600:.2f} h')

$ /content/c2venv/bin/python c2_xgb_confirm.py --processes 4 --n-jobs 2 --out /content/drive/MyDrive/c2_runs/T6
1280 fits requested, 1280 to run now, processes 4 x n_jobs 2, device cpu
BASE           seed  1  best_iter   88     8.1 s  additivity gap 3.2e-04
BASE           seed  3  best_iter   84     8.0 s  additivity gap 3.0e-04
BASE           seed  0  best_iter  101     9.1 s  additivity gap 4.7e-04
BASE           seed  2  best_iter  102     9.3 s  additivity gap 3.9e-04
BASE           seed  4  best_iter   98     9.4 s  additivity gap 4.0e-04
BASE           seed  5  best_iter  103     9.7 s  additivity gap 4.4e-04
BASE           seed  6  best_iter   93     9.0 s  additivity gap 4.1e-04
BASE           seed  7  best_iter  125    11.0 s  additivity gap 5.9e-04
BASE           seed  9  best_iter   89     8.8 s  additivity gap 4.3e-04
BASE           seed 10  best_iter   89     8.7 s  additivity gap 4.1e-04
BASE           seed  8  best_iter  121    10.9 s  additivity gap 5.8e-04
BASE        

In [8]:
# 8. The verdict, the T7-against-T6 comparison, and the results file for download
with open(f'{OUT}/c2x_verdict.txt') as f:
    print(f.read())
T6 = f'{DRIVE}/c2_runs/T6'
if RUN == 'T7' and os.path.isfile(f'{T6}/c2x_verdict.json'):
    def strip(path):
        with open(path) as f:
            v = json.load(f)
        v.pop('provenance')
        return v
    cmp = {f: rule(f'{OUT}/{f}') == rule(f'{T6}/{f}') for f in ('c2x_pooled.csv', 'c2x_city.csv')}
    cmp['c2x_predictions.npz'] = sha(f'{OUT}/c2x_predictions.npz') == sha(f'{T6}/c2x_predictions.npz')
    cmp['verdict apart from provenance'] = strip(f'{OUT}/c2x_verdict.json') == strip(f'{T6}/c2x_verdict.json')
    line = f'T7 against T6 (addendum A2.2): {cmp}'
    print(line)
    with open(f'{LOGS}/t7_vs_t6.txt', 'w') as f:
        f.write(line + '\n')
zpath = f'{DRIVE}/c2_runs/{RUN}_results.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
    for base, dirs, files in os.walk(OUT):
        dirs[:] = sorted(d for d in dirs if d != 'c2x_parts')    # the parts are inside c2x_predictions.npz
        for f in sorted(files):
            p = os.path.join(base, f)
            zf.write(p, os.path.relpath(p, OUT))
drive.flush_and_unmount()
print('saved', zpath, '- send this file')

C2 verdict (rules of C2_PREREGISTRATION.md sections 7-9)
pre-registration SHA-256 c3b38c294ba09b1f872e414d48e40623adb1f9d1382473099c2cc78494debd50
addendum A1 SHA-256 c2d32acd5cbe91dede7cd460ee0ceee2deb029755f0d58964e8d39915e134ed7
addendum A2 SHA-256 0f3e15ce2d78ed9dedf2c436f417c4c80e9a81c9e14de38a30c9a2854151348c (platform: Linux x86-64, n_jobs 2)

PRIMARY (MET_PP against the 10 POOL_PP sets, full test year; order H1 -> H2 -> H1b, one-sided 5%)
  H1 MAE (null mean - real)          effect +1.139 (90% +0.843 to +1.434) = +4.92% (+3.64 to +6.20)  p 0.0000  -> pass  below every null mean: True
                                     real 21.989, null mean 23.128, null-set s.d. 0.154, rel. SE of null mean 0.0021
  H2 short-band share excess (pp)    effect +3.474 (90% +3.327 to +3.621)  p 0.0000  -> pass
  H1b MAE (MET - MET_PP), Welch      effect +0.439 (90% +0.215 to +0.662)  p 0.0008  -> pass
  lockdown check: p 0.0000 -> no sentence

GATE: g = 4.92% against 13% -> option A not run

WORDIN